<p align="center"><a href="https://www.plus2net.com/python/file-append.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python File Append: Preserving Data with 'a' & 'a+' Modes Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-append.php](https://www.plus2net.com/python/file-append.php)

Run cells in order in Google Colab. All file appending concepts, append mode (`'a'`) vs read-and-append (`'a+'`), EOF cursor invariants, thread-safe timestamped event logging, structured CSV appending, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Basic File Appending with Mode 'a'
Mode `'a'` appends data to the end of an existing file without truncating or erasing previous contents.

In [ ]:
# Initial write
with open("demo_append.log", "w", encoding="utf-8") as f:
    f.write("Line 1: Initialized dataset.\n")

# Append mode attaches to the end
with open("demo_append.log", "a", encoding="utf-8") as f:
    f.write("Line 2: Appended transaction record.\n")

# Read back to verify both lines exist
with open("demo_append.log", "r", encoding="utf-8") as f:
    print("--- File Content ---")
    print(f.read())


### 2. Reading and Appending with Mode 'a+'
Mode `'a+'` allows both reading and appending. Reading immediately starts at EOF, so `seek(0)` is required to read earlier lines.

In [ ]:
with open("demo_append.log", "a+", encoding="utf-8") as fob:
    fob.write("Line 3: Appended via a+ mode.\n")
    
    # Immediate read returns empty string (cursor at EOF)
    print("Immediate read at EOF:", repr(fob.read()))
    
    # Rewind cursor to start
    fob.seek(0)
    all_records = fob.read()
    print("Cumulative records read via seek(0):\n" + all_records)


### 3. Demonstrating the seek() Invariant in Append Mode
Under standard OS semantics, all writes in mode `'a'` are placed at EOF, regardless of `seek()`.

In [ ]:
with open("demo_append.log", "a", encoding="utf-8") as f:
    f.seek(0)  # Attempt to move pointer to beginning
    f.write("Line 4: Written after seek(0).\n")  # Still lands at EOF!

with open("demo_append.log", "r", encoding="utf-8") as f:
    lines = f.readlines()
    print("First line:", repr(lines[0]))
    print("Last line: ", repr(lines[-1]))


### 4. Real-World Recipe: Timestamped Event Logger
Append structured timestamped messages to an application activity log.

In [ ]:
import datetime

def log_event(filepath: str, level: str, message: str) -> None:
    ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
    entry = f"[{ts}] [{level.upper()}] {message}\n"
    with open(filepath, "a", encoding="utf-8") as stream:
        stream.write(entry)

log_event("audit.log", "info", "User 102 logged in")
log_event("audit.log", "warning", "Failed API request to payment gateway")

with open("audit.log", "r", encoding="utf-8") as f:
    print(f.read())


### 5. Real-World Recipe: Structured CSV Appending
Append student records to a CSV file, writing the header only when creating a new file.

In [ ]:
import csv
import os

def append_student_record(csv_path: str, record: list) -> None:
    file_exists = os.path.exists(csv_path) and os.path.getsize(csv_path) > 0
    with open(csv_path, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        if not file_exists:
            writer.writerow(["ID", "Name", "Math", "Science", "English", "Total"])
        writer.writerow(record)

append_student_record("students.csv", [1, "Ravi", 20, 30, 40, 120])
append_student_record("students.csv", [2, "Raju", 30, 40, 50, 130])

with open("students.csv", "r", encoding="utf-8") as f:
    print(f.read())


### 6. Common Trap: Calling read() in Mode 'a'
Attempting to read from a file opened in mode `'a'` raises `io.UnsupportedOperation`.

In [ ]:
import io

try:
    with open("demo_append.log", "a") as f:
        f.read()
except io.UnsupportedOperation as err:
    print("Handled expected error:", err)


### 7. Self-Checking Verification Suite
Run unit assertions verifying append non-destruction, `a+` rewinding, and error handling.

In [ ]:
class AppendVerifier:
    @staticmethod
    def append_line(path: str, text: str) -> int:
        with open(path, "a", encoding="utf-8") as stream:
            return stream.write(text + "\n")

    @staticmethod
    def read_all_lines(path: str) -> list:
        with open(path, "r", encoding="utf-8") as stream:
            return [line.rstrip("\n") for line in stream]

# Run verification tests
v = AppendVerifier()
v.append_line("test_verify.log", "Alpha")
v.append_line("test_verify.log", "Beta")
assert v.read_all_lines("test_verify.log") == ["Alpha", "Beta"]

with open("test_verify.log", "a+", encoding="utf-8") as f:
    f.write("Gamma\n")
    f.seek(0)
    assert "Gamma" in f.read()

print("All file append unit assertions passed successfully! 100% verified.")
